# A local benchmark harness [Step 08.02 - Four pieces, four tasks]

> **MLCourse - Agentic AI - Agent Patterns**

We build the smallest thing that is honestly a benchmark: a fixed task set, a
deterministic environment, an instrumented agent, and a grader that shares no code
with the agent.

Four tasks. Two tools. Runs in a couple of minutes.

### What you'll learn

- How to structure a harness so the agent and the grader cannot contaminate each
  other.
- Why every task carries **its own** grader.
- Instrumenting steps, tool calls and tokens - because "did it pass" is only half
  the result.

### Key takeaways

- The grader is written **before** you look at any agent output. Otherwise you are
  fitting the grader to the agent.
- Record *how* it passed, not just *that* it passed. An agent that answers correctly
  without calling a tool guessed, and will not generalise.
- A run must never crash on one bad task. Catch, record, continue.

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                   # environment variables
import time                                 # timing + backoff sleeps
from pathlib import Path                    # locating the .env
from dotenv import load_dotenv              # reads KEY=value pairs from .env

# Walk UP from this notebook until we find the folder that CONTAINS the track
# directory `03_agentic_ai` (that folder is the repo root), then load the
# gitignored .env that lives INSIDE the track.
#
# Pitfall worth naming: it is easy to write the walk so that it stops at the
# repo root and then load `ROOT/.env`, which does not exist - `load_dotenv`
# returns False and says nothing, so the notebook silently has no key.
ROOT = Path.cwd()
while not (ROOT / "03_agentic_ai").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ENV_PATH = ROOT / "03_agentic_ai" / ".env"
load_dotenv(ENV_PATH)

GROQ_MODEL = "qwen/qwen3.8-27b"             # the one hosted model this course uses
GROQ_KEY = os.environ["GROQ_API_KEY"]       # KeyError here = .env not found. Never print it.

# A local Ollama model (e.g. `llama3.1:8b`) is a perfectly good substitute if you
# have no Groq key - swap the two lines in `make_llm`. We deliberately do NOT
# write a silent fallback branch: a notebook that quietly changes model behind
# your back produces numbers you cannot trust.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 256):
    """Return the chat model used everywhere in this module."""
    return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                    temperature=temperature, max_tokens=max_tokens)


PACE = 0.7          # seconds to wait between calls: the free tier is 8000 TPM


def safe_invoke(model, messages, retries: int = 5, pause: float = 2.0):
    """Invoke a chat model, backing off exponentially on 429 / rate-limit errors.

    Returns the AIMessage. Raises if every retry is exhausted - we want a loud
    failure, not a quiet wrong number.
    """
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(PACE)                       # pace the next call
            return out
        except Exception as exc:                   # noqa: BLE001 - we re-raise below
            text = str(exc).lower()
            if "429" in text or "rate" in text or "quota" in text:
                wait = pause * (2 ** attempt)
                print("  [rate limit] sleeping %.1fs (attempt %d/%d)" % (wait, attempt + 1, retries))
                time.sleep(wait)
                continue
            raise
    raise RuntimeError("rate limited after %d attempts" % retries)


# Published Groq list price for this model at the time of writing, in USD per
# 1M tokens. Substitute your own numbers - the METHOD is the lesson, not these
# two constants.
PRICE_IN_PER_M = 0.29
PRICE_OUT_PER_M = 0.59


def usd(in_tok: int, out_tok: int) -> float:
    """Convert a token count into dollars at the prices above."""
    return in_tok / 1e6 * PRICE_IN_PER_M + out_tok / 1e6 * PRICE_OUT_PER_M


print("env file :", ENV_PATH, "(exists:", ENV_PATH.exists(), ")")
print("model    :", GROQ_MODEL)
print("key      : loaded, %d chars" % len(GROQ_KEY))


In [2]:
PACE = 3.0
print("PACE =", PACE)

PACE = 3.0


### The benchmark: world, tools, tasks, grader


In [ ]:
# Four pieces, and they must be separable. If the grader lives inside the agent,
# you are not running a benchmark, you are running a demo.

import re

# 1. THE WORLD -----------------------------------------------------------------
EMPLOYEES = {
    "priya":  dict(name="Priya",  dept="Engineering", salary=82000),
    "marco":  dict(name="Marco",  dept="Engineering", salary=61000),
    "ana":    dict(name="Ana",    dept="Design",      salary=74000),
    "kofi":   dict(name="Kofi",   dept="Engineering", salary=69000),
    "lena":   dict(name="Lena",   dept="Design",      salary=71000),
}

TOOL_CALLS = {"n": 0}


# 2. THE TOOLS -----------------------------------------------------------------
def tool_lookup(arg):
    """lookup(name) -> the employee record, or an error string."""
    TOOL_CALLS["n"] += 1
    rec = EMPLOYEES.get(arg.strip().lower().strip('"\''))
    if not rec:
        return "ERROR: no employee named %r" % arg
    return "name=%s dept=%s salary=%d" % (rec["name"], rec["dept"], rec["salary"])


def tool_count_dept(arg):
    """count_dept(department) -> how many people work in it."""
    TOOL_CALLS["n"] += 1
    d = arg.strip().lower().strip('"\'')
    n = sum(1 for r in EMPLOYEES.values() if r["dept"].lower() == d)
    return "count=%d" % n


TOOLS = {"lookup": tool_lookup, "count_dept": tool_count_dept}

TOOL_DOC = """You have exactly two tools:
  lookup(name)            -> name, department and salary of one employee
  count_dept(department)  -> how many people work in that department

To use a tool, reply with ONLY one line:
CALL: <tool_name>(<argument>)

When you know the answer, reply with ONLY one line:
FINAL: <answer>

Never output both. Never explain. One line per turn."""


# 3. THE TASKS -----------------------------------------------------------------
# Each task carries its OWN grader. Different tasks need different checks, and
# pretending otherwise is how benchmarks end up measuring string formatting.

def num_grader(expected, tol=0.5):
    def check(answer):
        nums = re.findall(r"-?\d+(?:\.\d+)?", (answer or "").replace(",", ""))
        if not nums:
            return False
        return abs(float(nums[-1]) - expected) <= tol
    return check


def word_grader(expected):
    def check(answer):
        return expected.lower() in (answer or "").lower()
    return check


TASKS = [
    dict(id="salary",   q="What is Priya's salary?",
         grade=num_grader(82000), min_tools=1),
    dict(id="headcount", q="How many people work in Engineering?",
         grade=num_grader(3), min_tools=1),
    dict(id="combined", q="What is the combined salary of Priya and Marco? Give a single number.",
         grade=num_grader(143000), min_tools=2),
    dict(id="compare",  q="Who earns more, Ana or Marco? Answer with only the name.",
         grade=word_grader("ana"), min_tools=2),
]

print("%d tasks, %d tools, %d employees" % (len(TASKS), len(TOOLS), len(EMPLOYEES)))


### On the graders

`num_grader` takes the **last** number in the answer and compares within a
tolerance. `word_grader` does a normalised substring check. Both are boring on
purpose - that is what "deterministic" buys you.

`min_tools` records how many tool calls the task genuinely requires. It is not used
for grading; it is used to catch **lucky guesses**, which are the most common way a
small benchmark lies to you.

### 1. The agent under test

A minimal ReAct-shaped loop. The model gets a one-line protocol and either calls a
tool or answers. Capped at three steps.

### The agent under test


In [ ]:
# A minimal ReAct-shaped loop: the model either calls a tool or answers. Capped at
# MAX_STEPS, because an unbounded agent in a benchmark is an unbounded bill.

MAX_STEPS = 3
CALL_RE = re.compile(r"CALL:\s*(\w+)\s*\((.*?)\)", re.S)
FINAL_RE = re.compile(r"FINAL:\s*(.+)", re.S)


def run_agent(question, temperature=0.0, verbose=False):
    """Run one attempt. Returns a dict - never raises, so one bad task cannot
    abort a benchmark run halfway through."""
    llm = make_llm(temperature=temperature, max_tokens=120)
    TOOL_CALLS["n"] = 0
    transcript = []
    messages = [("system", "You are a precise data assistant.\n\n" + TOOL_DOC),
                ("user", question)]
    tin = tout = 0
    answer = None
    steps = 0

    for steps in range(1, MAX_STEPS + 1):
        msg = safe_invoke(llm, messages)
        u = msg.usage_metadata or {}
        tin += u.get("input_tokens", 0)
        tout += u.get("output_tokens", 0)
        text = msg.content.strip()
        transcript.append(text)
        if verbose:
            print("  [step %d] %s" % (steps, text.replace("\n", " ")[:100]))

        fin = FINAL_RE.search(text)
        call = CALL_RE.search(text)
        # A model that emits both is ambiguous; prefer the tool call, since a
        # premature FINAL is the more common failure.
        if call and (not fin or call.start() < fin.start()):
            tool, arg = call.group(1), call.group(2)
            result = TOOLS[tool](arg) if tool in TOOLS else "ERROR: no such tool %r" % tool
            if verbose:
                print("           -> %s" % result)
            messages = messages + [("assistant", text), ("user", "TOOL RESULT: " + result)]
            continue
        if fin:
            answer = fin.group(1).strip().splitlines()[0].strip()
            break
        # Neither: nudge once, then give up.
        messages = messages + [("assistant", text),
                               ("user", "Reply with ONLY one line: CALL: ... or FINAL: ...")]

    return dict(answer=answer, steps=steps, tool_calls=TOOL_CALLS["n"],
                tokens=tin + tout, tin=tin, tout=tout, transcript=transcript)


Three details in that loop that matter more than they look:

- **`MAX_STEPS`.** An unbounded agent loop in a benchmark is an unbounded bill.
- **`run_agent` never raises.** A benchmark that dies on task 3 of 4 gives you
  nothing. Record the failure and continue.
- **`TOOL_CALLS` resets per attempt.** This is the "reset the environment" rule from
  notebook 01, in its smallest possible form.

### 2. One task, verbosely

Watch the protocol working before trusting it in a loop.

In [5]:
r = run_agent(TASKS[2]["q"], verbose=True)        # combined salary - needs 2 lookups
print()
print("answer     :", r["answer"])
print("steps      :", r["steps"])
print("tool calls :", r["tool_calls"])
print("tokens     :", r["tokens"])
print("graded     :", "PASS" if TASKS[2]["grade"](r["answer"]) else "FAIL")

  [step 1] CALL: lookup(Priya)
           -> name=Priya dept=Engineering salary=82000


  [step 2] CALL: lookup(Marco)
           -> name=Marco dept=Engineering salary=61000


  [step 3] FINAL: 143000

answer     : 143000
steps      : 3
tool calls : 2
tokens     : 536
graded     : PASS


### 3. A full run


In [6]:
def run_benchmark(temperature=0.0, verbose=False, label=""):
    """Run every task once. Returns a list of result rows."""
    rows = []
    for t in TASKS:
        r = run_agent(t["q"], temperature=temperature, verbose=verbose)
        passed = bool(t["grade"](r["answer"]))
        suspicious = passed and r["tool_calls"] < t["min_tools"]
        rows.append(dict(id=t["id"], passed=passed, answer=r["answer"],
                         steps=r["steps"], tool_calls=r["tool_calls"],
                         tokens=r["tokens"], suspicious=suspicious))
        print("%s%-11s %-5s answer=%-22s steps=%d tools=%d tokens=%d%s"
              % (label, t["id"], "PASS" if passed else "FAIL",
                 str(r["answer"])[:22], r["steps"], r["tool_calls"], r["tokens"],
                 "  <- GUESSED (fewer tools than required)" if suspicious else ""))
    return rows


run1 = run_benchmark(temperature=0.0)

salary      PASS  answer=82000                  steps=2 tools=1 tokens=302


headcount   PASS  answer=3                      steps=2 tools=1 tokens=287


combined    PASS  answer=143000                 steps=3 tools=2 tokens=536


compare     PASS  answer=Ana                    steps=3 tools=2 tokens=519


In [7]:
def summarise(rows, name):
    n = len(rows)
    p = sum(1 for r in rows if r["passed"])
    print("%-14s score %d/%d = %.3f | %d tokens | %d tool calls | %d suspicious"
          % (name, p, n, p / n, sum(r["tokens"] for r in rows),
             sum(r["tool_calls"] for r in rows),
             sum(1 for r in rows if r["suspicious"])))
    return p / n


score1 = summarise(run1, "run 1 (T=0)")

run 1 (T=0)    score 4/4 = 1.000 | 1644 tokens | 6 tool calls | 0 suspicious


### 4. The number you just produced is a sample, not a score

This is the single most important idea in the module.

You now have a figure like "0.750". It is tempting to write it down and compare it
against tomorrow's. But that number came from **one run of a stochastic system**.
Even at temperature 0, hosted inference is not bit-reproducible: batching, hardware
and server-side model updates all introduce variation.

Prove it to yourself - run it again, unchanged.

In [8]:
run2 = run_benchmark(temperature=0.0)
score2 = summarise(run2, "run 2 (T=0)")

print()
print("run 1: %.3f    run 2: %.3f    difference: %+.3f" % (score1, score2, score2 - score1))
same = [r1["id"] for r1, r2 in zip(run1, run2) if r1["passed"] != r2["passed"]]
if same:
    print("tasks that changed verdict between identical runs:", same)
else:
    print("no task changed verdict - but note that 4 tasks x 2 runs cannot")
    print("detect flakiness below roughly 12%.")

salary      PASS  answer=82000                  steps=2 tools=1 tokens=302


headcount   PASS  answer=3                      steps=2 tools=1 tokens=287


combined    PASS  answer=143000                 steps=3 tools=2 tokens=536


compare     PASS  answer=Ana                    steps=3 tools=2 tokens=519
run 2 (T=0)    score 4/4 = 1.000 | 1644 tokens | 6 tool calls | 0 suspicious

run 1: 1.000    run 2: 1.000    difference: +0.000
no task changed verdict - but note that 4 tasks x 2 runs cannot
detect flakiness below roughly 12%.


In [9]:
print("token cost of one benchmark run")
print("-" * 46)
tok = sum(r["tokens"] for r in run1)
print("tokens per run      : %d" % tok)
print("cost per run        : $%.6f" % usd(sum(r["tokens"] for r in run1) // 2,
                                          sum(r["tokens"] for r in run1) // 2))
print("cost of 100 runs    : $%.4f" % (100 * usd(tok // 2, tok // 2)))
print()
print("Scale that to a real task set: 200 tasks x 5 repeats is 250x this run.")
print("Benchmark cost is the reason people report single runs. It is not a")
print("good reason.")

token cost of one benchmark run
----------------------------------------------
tokens per run      : 1644
cost per run        : $0.000723
cost of 100 runs    : $0.0723

Scale that to a real task set: 200 tasks x 5 repeats is 250x this run.
Benchmark cost is the reason people report single runs. It is not a
good reason.


### Pitfalls

- **Writing the grader after seeing the output.** You will unconsciously widen it
  until the agent passes. Write it first; if it is wrong, fix it *and re-run
  everything*.
- **Sharing normalisation code between agent and grader.** If both call the same
  `clean_answer()`, a bug in it is invisible.
- **No lucky-guess check.** Our `suspicious` flag is crude but it catches the
  commonest small-benchmark lie: an agent that "knows" the answer without looking.
- **Letting one task crash the run.** Catch, record `passed=False`, continue.
- **Not versioning the task set.** If you add tasks, scores before and after are not
  comparable. Version the set like code.

### Next

Notebook 03 replaces the single number with **pass@k**, and notebook 04 gives it
an error bar.